# Zero-shot LLM control of the FFW robot

Can a frontier LLM, driving the robot through API tool calls and seeing only the camera images and proprioception, do a pick-and-place zero-shot? The task is the tutorial's pick-and-place with one change: a 5 cm red cube replaces the mug (the FFW gripper cannot get around the mug from above, which is how the robot starts in its real-robot table-top pose), and the plate sits on the left arm's side of the table.

This notebook uses [inspect-robots](https://github.com/robocurve/inspect-robots) (vendored as a submodule in `inspect-robots/`) as the harness:

- **Embodiment** `ffw_mujoco` ([ffw_inspect/embodiment.py](ffw_inspect/embodiment.py)): the tutorial's MuJoCo scene with the cube ([asset/example_scene_llm.xml](asset/example_scene_llm.xml)), rendered headlessly, starting from the real robot's table-top pose (`ai_worker` `tabletop_manip_position.yaml`). The action is a 14-D end-effector displacement (x, y, z, roll, pitch, yaw, gripper per arm) in the world frame; IK and gravity compensation run inside the embodiment.
- **Task** `ffw-cube-on-plate` ([ffw_inspect/task.py](ffw_inspect/task.py)): seeded layouts (seed 0 nominal, later seeds jittered by 2 cm) with a privileged success check (cube on the plate, grippers open, arms retracted).
- **Policy** `agent` (the `inspect-robots-agent` plugin): the LLM gets `move_by`, `take_pic`, `done` and `give_up` tools built from the embodiment's declared spaces, plus the embodiment's operating notes in its system prompt. Every call, image and token count is recorded.

## Setup

Run this in the `lerobot_mujoco` env. The packages are already installed there (`pip install -e inspect-robots -e inspect-robots/plugins/inspect-robots-agent -e ffw_inspect`).

Put your API key in a `.env` file in the repo root (see `.env.example`):

```
OPENAI_API_KEY=sk-...
```


In [ ]:
import os, sys, json, time
from pathlib import Path

os.environ.setdefault("MUJOCO_GL", "egl")          # headless rendering, no display needed
REPO = Path.cwd()
assert (REPO / "ffw_inspect").exists(), "run this notebook from the repo root"

from inspect_robots._dotenv import init_dotenv
init_dotenv(os.environ, REPO / ".env")              # loads OPENAI_API_KEY etc. from .env
assert os.environ.get("OPENAI_API_KEY"), "put OPENAI_API_KEY in .env (see .env.example)"

import numpy as np
from inspect_robots import eval
from inspect_robots.approver import ChainApprover, ClampApprover, DeltaLimitApprover
from inspect_robots_agent import LLMAgentPolicy
from ffw_inspect import FFWMujocoEmbodiment, FFWOraclePolicy, ffw_cube_on_plate
print("ok")

## Configuration

- `MODEL`: OpenRouter-style `provider/model`. GPT models with reasoning need the Responses wire.
- One episode (`n_scenes=1`), seed 0, which is the layout the ACT demos were collected in.
- `max_llm_calls` bounds the number of tool-call rounds; `max_steps` bounds sim time (1200 steps = 60 s at 20 Hz).
- `privileged_state=False`: the model sees only images and proprioception. Set it to `True` to also give it the cube and plate positions (ablation).

In [ ]:
MODEL = "openai/gpt-6.1-sol"
WIRE = "responses"           # GPT reasoning models need /responses for function tools
EFFORT = "medium"
MAX_LLM_CALLS = 60
PRIVILEGED = False
LOG_DIR = "logs"

task = ffw_cube_on_plate(n_scenes=1, max_steps=1200)
embodiment = FFWMujocoEmbodiment(privileged_state=PRIVILEGED)
guardrails = ChainApprover(ClampApprover(embodiment.info.action_space), DeltaLimitApprover(embodiment.info.action_space))
print(embodiment.info.docs)

## Sanity check: the scripted oracle

Before spending API calls, run the privileged scripted policy through the same harness (it succeeds in about 270 steps). If it does not succeed, the embodiment is broken, not the LLM.

In [ ]:
oracle_emb = FFWMujocoEmbodiment(privileged_state=True)
(oracle_log,) = eval(task, FFWOraclePolicy(), oracle_emb, log_dir=LOG_DIR, approver=guardrails)
oracle_emb.close()
print(oracle_log.status, oracle_log.results.metrics)
assert oracle_log.results.metrics.get("success_at_end") == 1.0, "oracle failed: fix the embodiment first" 

## Run the LLM agent

`transcript_echo=True` prints the conversation live: the model's notes on what it sees and why it moves, each tool call, and each tool result. Expect several minutes (one API call per motion, plus sim time).

In [ ]:
policy = LLMAgentPolicy(model=MODEL, wire=WIRE, effort=EFFORT, max_llm_calls=MAX_LLM_CALLS, transcript_echo=True, images="always")
t0 = time.time()
(log,) = eval(task, policy, embodiment, log_dir=LOG_DIR, approver=guardrails, store_frames=True)
print(f"\nstatus: {log.status}   metrics: {log.results.metrics}   wall: {time.time()-t0:.0f}s")
for scene in log.samples:
    print(scene.scene_id, scene.status, "termination:", scene.termination_reasons, "llm_usage:", [m.get("llm_usage") for m in scene.trial_metadata])
    for m in scene.trial_metadata:
        if m.get("hindsight"): print("hindsight:", m["hindsight"])

## Inspect the run

Every run writes an `EvalLog` JSON under `logs/` with the full transcript, stored frames, and the executed actions. The CLI renders it as HTML.

In [ ]:
log_path = sorted(Path(LOG_DIR).glob("ffw-cube-on-plate_*.json"), key=os.path.getmtime)[-1]
print(log_path)
# Transcript as text (what the model saw and said), or open the HTML report:
#   inspect-robots inspect <log> --transcript
#   inspect-robots view <log>
!PYTHONNOUSERSITE=1 inspect-robots inspect "$log_path" --transcript | head -120

## Render the rollout

`eval(..., store_frames=True)` saved every camera frame of every step under `logs/frames/<run>/` as `.npy`, the executed actions under `logs/actions/`, and the agent plugin saved every raw API request and response under `logs/wire/`. Stitch the agentview frames into a GIF here, or run `inspect-robots video <log.json>` for per-camera MP4s.

In [ ]:
import re, numpy as np, imageio.v2 as imageio
# Stored frames are .npy arrays named ~f1~<scene>-e<epoch>~<camera>_<step>.npy under logs/frames/<run>/
run_dir = sorted((Path(LOG_DIR) / "frames").iterdir(), key=os.path.getmtime)[-1]
frames = sorted(run_dir.glob("*~agentview_*.npy"), key=lambda p: int(re.search(r"_(\d+)\.npy$", p.name).group(1)))
print(run_dir.name, len(frames), "agentview frames")
if frames:
    out = Path("media") / "llm_rollout.gif"
    imageio.mimsave(out, [np.load(p) for p in frames[::2]], duration=0.1, loop=0)
    print("wrote", out)
# Per-camera MP4s instead:  inspect-robots video <log.json>

## Same run from the CLI

```
PYTHONNOUSERSITE=1 inspect-robots run --task ffw-cube-on-plate --policy agent --embodiment ffw_mujoco \
    -P model=openai/gpt-6.1-sol -P wire=responses -P effort=medium -P max_llm_calls=60 \
    -P transcript_echo=true --store-frames
inspect-robots view logs/
```

Switching models is a flag: `-P model=anthropic/claude-fable-5-1 -P wire=messages`, or `-P model=google/gemini-3.7-flash -P wire=interactions`.